In [0]:
# spark.sql("create schema sales_project_streaming.brz;")
# spark.sql("create schema sales_project_streaming.slv;")
# spark.sql("create schema sales_project_streaming.gld;")

In [0]:
# %sql
# create volume sales_project_streaming.slv.checkpoints_vol;

In [0]:
topics = {"sales": "sales_stream", 
          "employees": "employees_stream", 
          "expenses": "expenses_stream", 
          "regions": "regions_stream"}

checkpoints_brz = {"sales": "/Workspace/Shared/checkpoints_streaming/sales_chck_brz",
               "employees": "/Workspace/Shared/checkpoints_streaming/employees_chck_brz",
               "expenses": "/Workspace/Shared/checkpoints_streaming/expenses_chck_brz",
               "regions": "/Workspace/Shared/checkpoints_streaming/regions_chck_brz"}

checkpoints_slv = {"sales": "/Volumes/sales_project_streaming/slv/checkpoints_vol/sales_chck_slv_v2/",
               "employees": "/Volumes/sales_project_streaming/slv/checkpoints_vol/employees_chck_slv/",
               "expenses": "/Volumes/sales_project_streaming/slv/checkpoints_vol/expenses_chck_slv/",
               "regions": "/Volumes/sales_project_streaming/slv/checkpoints_vol/regions_chck_Slv/"}

ngrokip = "0.tcp.in.ngrok.io:15585"

In [0]:
# %sql
# create or replace table sales_project_streaming.brz.sales(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.employees(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.expenses(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.regions(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

In [0]:
from pyspark.sql import functions as F

df = (spark.readStream.format("kafka")
      .option("kafka.bootstrap.servers", ngrokip)
      .option("subscribe", "sales_stream")
      .option("startingOffsets", "earliest")
      .option("failOnDataLoss", "false")
      .load())

df = df.selectExpr("cast(key as string) as key",
                   "cast(value as string) as value",
                   "topic",
                   "partition",
                   "offset",
                   "timestamp").withColumn("ingestion_ts", F.current_timestamp())

query = (df.writeStream.format("delta")
         .option("checkpointLocation", "/Workspace/Shared/checkpoints_streaming/sales_chck_brz")
         .trigger(availableNow = True)
         .outputMode("append")
         .table("sales_project_streaming.brz.sales"))

query.awaitTermination()

In [0]:
from pyspark.sql.types import *

df = (spark.readStream
      .format("delta")
      .table("sales_project_streaming.brz.sales")
      )

schema = StructType([
    StructField("sales_id", LongType(), False),
    StructField("employee_id", LongType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", LongType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

parsed_df = df.withColumn("parsed_value", F.from_json("value", schema)).select("parsed_value", "ingestion_ts")

normalised_df = parsed_df.select("parsed_value.sales_id", "parsed_value.employee_id", "parsed_value.region_id",
                                 "parsed_value.product_id", "parsed_value.quantity", F.col("parsed_value.sales_amount").alias("amount"),
                                 "parsed_value.event_time")

normalised_df = (normalised_df.withWatermark("event_time", "10 minutes").dropDuplicates(subset = ["sales_id"])
                 .withColumn("hour", F.hour(F.col("event_time")))
                 .withColumn("event_date", F.to_date(F.col("event_time")))
                 .withColumn("day", F.date_format(F.col("event_time"), "EEEE"))
                 .withColumn("day_of_week", F.dayofweek(F.col("event_time")))
                 .withColumn("is_weekend", F.col("day_of_week").isin([7,1]))
                 .withColumn("week_of_month", 
                             F.weekofyear(F.col("event_date")) - F.weekofyear(F.date_sub(F.col("event_date"), F.dayofmonth(F.col("event_date"))+1))+1)
                 )

good_df = (normalised_df.filter((F.col("sales_id").isNotNull()) & (F.col("employee_id").isNotNull()) &
                                (F.col("region_id").isNotNull()) & (F.col("quantity").isNotNull()) &
                                (F.col("amount").isNotNull()) & (F.col("amount")>=0))
           .withColumn("processed_time", F.current_timestamp()))

good_query = (good_df.writeStream
         .format("delta")
         .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/sales_chck_slv_v2/")
         .trigger(availableNow = True)
         .outputMode("append")
         .partitionBy("event_date")
         .toTable("sales_project_streaming.slv.sales")
         )

bad_df = (normalised_df.filter((F.col("sales_id").isNull()) | (F.col("employee_id").isNull()) |
                                (F.col("region_id").isNull()) | (F.col("quantity").isNull()) |
                                (F.col("amount").isNull()) | (F.col("amount")<0))
           .withColumn("processed_time", F.current_timestamp()))

bad_query = (bad_df.writeStream
         .format("delta")
         .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/bad_sales_chck_slv_v2/")
         .trigger(availableNow = True)
         .outputMode("append")
         .toTable("sales_project_streaming.brz.bad_records_sales")
         )

good_query.awaitTermination()
bad_query.awaitTermination()

In [0]:
%sql
optimize sales_project_streaming.slv.sales
zorder by (region_id);

vacuum sales_project_streaming.slv.sales;

In [0]:
display(spark.sql("select * from sales_project_streaming.slv.sales"))